<br>

# Notebook 2 - Crop Simulation

<hr>
This key module simulates all the possible crop cycles to  find the best crop cycle that produces maximum yield for a particular grid. During the simulation process for each grid, 365 crop cycle simulations are performed. Each simulation corresponds to cycles that start from each day of the year (starting from Julian date 0 to Julian date 365). Similarly, this process is performed by the program for each grid in the study area.

Prepared by Geoinformatics Center, AIT
<hr>


# ── Google Colab setup ──────────────────────────────────────────────
# Run these steps ONLY when using Google Colab.

# Step 1: Clone the repository (first time only)
# !git clone https://github.com/YOUR_USERNAME/PK-PyAEZ.git /content/PK-PyAEZ

# Step 2 (optional): Mount Google Drive to persist outputs across sessions
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:


# Step 2 (optional): Mount Google Drive to persist outputs across sessions
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys, os
repo = '/content/drive/MyDrive/PK-PyAEZ'
if repo not in sys.path:
    sys.path.insert(0, repo)
os.chdir(repo)

Then, installing any additional python packages that required to run PyAEZ.
If working on your own PC/machine, these additional installation will vary depending on what is already installed in your Python library.

In [ ]:
import sys

if 'google.colab' in sys.modules:
    import subprocess, os as _os
    subprocess.run(['apt-get', 'install', '-y', '-q', 'gdal-bin', 'libgdal-dev', 'python3-gdal'],
                   capture_output=True)
    _ver = subprocess.check_output(['gdal-config', '--version']).decode().strip()
    subprocess.run(['pip', 'install', '-q',
                    f'GDAL=={_ver}', 'numba', 'openpyxl', 'rasterio', 'requests', 'scipy', 'pandas'],
                   capture_output=True)
    subprocess.run(['pip', 'install', '-q', '-e', '/content/drive/MyDrive/PK-PyAEZ'], capture_output=True)
    print(f"Colab setup complete — GDAL {_ver}")

Now, we will import the specific Python packages we need for PyAEZ.

In [ ]:
'''import supporting libraries'''
import numpy as np
import matplotlib.pyplot as plt
import os
try:
    from osgeo import gdal
except:
    import gdal
gdal.UseExceptions()
import sys

Setting the working directory -- where our PyAEZ project is located.

In [ ]:
'Set the working directory'
import sys, os

if 'google.colab' in sys.modules:
    work_dir = '/content/drive/MyDrive/PK-PyAEZ'
else:
    work_dir = '..'

os.chdir(work_dir)
sys.path.insert(0, os.getcwd())
os.getcwd()

Check and create data output folder

In [6]:
import os
folder_path = './data_output/NB2/'
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print("Folder created successfully.")
else:
    print("Folder already exists.")


Folder already exists.


<hr>

## MODULE 2: CROP SIMULATION
Now, we will start executing the routines in Module 2


First, we initiate Module 2 Class instance by invoking the following commands:

In [7]:
# Import Module 2 and initate Class intance

from pyaez import CropSimulation, UtilitiesCalc
aez = CropSimulation.CropSimulation()
obj_util = UtilitiesCalc.UtilitiesCalc()

### Importing the climate dataset and the geographical data/rasters.

The package expects six climate variables, as daily or monthly observations, as Numpy arrays.
Arrays must be 3-dimensional, with the third axes containing the time dimension.
Unit of measures are expected as follows:
- Minimum temperature = Degree Celsius
- Maximum temperature = Degree Celsius
- Precipitation = Accumulated mm / day (or per month)
- Solar radiation = W/m^2
- Wind speed = Average m/s
- Relative humidity = Average fraction (0 to 1)

In addition to climate data, the system requires:
- A binary admin_mask, with 0 and 1 values. 0 pixels values will be not executed, while 1 pixels values will be executed
- An elevation layer

  

**All the datasets must have the same shape.**


## Reading Data

In [8]:
'''reading climate data'''
# Importing the climate data
max_temp = np.load('./data_input/climate/max_temp.npy')# maximum temperature
min_temp = np.load('./data_input/climate/min_temp.npy')  # minimum temperature
precipitation = np.load('./data_input/climate/precipitation.npy')  # precipitation
rel_humidity = np.load('./data_input/climate/relative_humidity.npy') # relative humidity
wind_speed = np.load('./data_input/climate/wind_speed.npy')# wind speed measured at two meters
short_rad = np.load('./data_input/climate/short_rad.npy') # shortwave radiation

# Load the geographical data/rasters
mask_path = './data_input/PK_Admin.tif'
mask = gdal.Open(mask_path).ReadAsArray()
elevation = gdal.Open('./data_input/PK_Elevation.tif').ReadAsArray()


/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


This section contains parameters that can be modified by the user:
- lat_min = minimum latitude of analysis
- lat_max = maximum latitude of analysis
- mask_value = the value in the admin_mask to exclude from the analysis (typically 0)
- daily = whether climate input data are daily (True) or monthly (False)

In [9]:
# Define the Area-Of-Interest's geographical extents
lat_min = 23.0   # Pakistan southern boundary
lat_max = 37.5   # Pakistan northern boundary
mask_value = 0  # pixel value in admin_mask to exclude from the analysis
daily = False #Type of climate data = True: daily, False: monthly

### Loading the imported data into the Object Class ('*aez*' Class)

In [10]:
aez.setStudyAreaMask(mask, mask_value)
aez.setLocationTerrainData(lat_min, lat_max, elevation)
if daily:
    aez.setDailyClimateData(
        min_temp, max_temp, precipitation, short_rad, wind_speed, rel_humidity)
else:
    aez.setMonthlyClimateData(
        min_temp, max_temp, precipitation, short_rad, wind_speed, rel_humidity)


### Setting up the crop parameter/ crop cycle parameter and soil water parameters (Mandatory)

In [11]:
# setting up the crop parameters, crop cycle and soil water parameters ***mandatory step
# New function, reading crop-specific biomass/yield loss/TSUM screening factors from excel sheet, xlsx file.
aez.readCropandCropCycleParameters(file_path = './data_input/input_crop_TSUM_parameters_maiz_sugar.xlsx',
                                   crop_name = 'maize')


aez.setSoilWaterParameters(Sa= 100*np.ones((mask.shape)), pc=0.5)

### Setting up the thermal screening parameters (Optional)

In [12]:
# If you're simulating perennial crops, this thermal screening is mandatory
# Compute Thermal Climate
tclimate = aez.getThermalClimate()

# Compute permafrost
permafrost_eval = aez.AirFrostIndexandPermafrostEvaluation()
frost_index = permafrost_eval[0]
permafrost_class = permafrost_eval[1]
# tclimate = gdal.Open("./data_output/NB1/PK_ThermalClimate.tif").ReadAsArray()# User to change this TClimate file
# permafrost_class = gdal.Open("./data_output/NB1/PK_permafrost.tif").ReadAsArray()# User to change this permafrost file

# Thermal Climate screening
aez.setThermalClimateScreening(tclimate, no_t_climate=[2,6,7,8,9,10,11,12])

# New Thermal Screening: Permafrost Screening
aez.setPermafrostScreening(permafrost_class= permafrost_class)

# Updated Temperature Profile screenign routine
aez.setCropSpecificRule(file_path = './data_input/crop-specific_rule_maiz_sugar.xlsx',
                         crop_name = 'maize')

# Is the crop perennial? (True/False)
is_perennial = True

/content/drive/MyDrive/PK-PyAEZ/PK-PyAEZ/pyaez/CropSimulation.py:1223: RuntimeWarning: invalid value encountered in divide
  fi = np.sqrt(ddf)/(np.sqrt(ddf) + np.sqrt(ddt))


### Importing LGP and LGPT Layers into the object Class (New function)

In [13]:
import os, sys
repo = '/content/drive/MyDrive/PK-PyAEZ/PK-PyAEZ'
os.chdir(repo)

# Verify NB1 outputs exist
for f in ['data_output/NB1/PK_LGP.tif', 'data_output/NB1/PK_LGPt5.tif', 'data_output/NB1/PK_LGPt10.tif']:
    print(f, "✓" if os.path.exists(f) else "MISSING")

data_output/NB1/PK_LGP.tif ✓
data_output/NB1/PK_LGPt5.tif ✓
data_output/NB1/PK_LGPt10.tif ✓


In [14]:
# Create climatic indicators independently in this notebook
# lgpt5 = aez.getThermalLGP5()
# lgpt10 = aez.getThermalLGP10()
# lgp = aez.getLGP(Sa=100, D=1) #has to be after LGPt are computed

# Or if you have the agro-climatic indicators calculated in Module I, you can use them.
lgp = gdal.Open('./data_output/NB1/PK_LGP.tif').ReadAsArray()
lgp5 = gdal.Open('./data_output/NB1/PK_LGPt5.tif').ReadAsArray()
lgp10 = gdal.Open('./data_output/NB1/PK_LGPt10.tif').ReadAsArray()

aez.ImportLGPandLGPT(lgp = lgp, lgpt5 = lgp5, lgpt10= lgp10)

### Simulate crop cycle


In [15]:
'''run simulations'''
aez.simulateCropCycle( start_doy=1, end_doy=365, step_doy=1, leap_year=False) # results are in kg / hectare

KeyboardInterrupt: 

### Maximum Attainable Yield for Rainfed and Irrigated Condition/ Optimum starting date


Yield Classification
1.   not suitable (yields between 0% and 20%)
2.   marginally suitable (yields between 20% and 40%)
3.   moderately suitable (yields between 40% and 60%)
4. suitable (yields between 60% and 80%)
5. very suitable (yields are equivalent to 80% or more of the overall maximum yield)



In [ ]:
# Now, showing the estimated and highly obtainable yield of a particular crop, results are in kg / hectare
yield_map_rain = aez.getEstimatedYieldRainfed()  # for rainfed
yield_map_irr = aez.getEstimatedYieldIrrigated()  # for irrigated

# Optimum cycle start date, the date when the highest yield are produced referenced from the start of crop cycle
starting_date_rain = aez.getOptimumCycleStartDateRainfed()
starting_date_irr = aez.getOptimumCycleStartDateIrrigated()

## get classified output of yield
# yield_map_rain_class = obj_util.classifyFinalYield(yield_map_rain)
# yield_map_irr_class = obj_util.classifyFinalYield(yield_map_irr)


In [ ]:
'''visualize result'''

"""Yield Maps"""
plt.imshow(yield_map_rain, vmax = np.max([yield_map_irr, yield_map_rain]), vmin = 0)
plt.colorbar()
plt.title('Rainfed Yield')
plt.show()

plt.imshow(yield_map_irr, vmax = np.max([yield_map_irr, yield_map_rain]), vmin = 0)
plt.colorbar()
plt.title('Irrigated Yield')
plt.show()

"""Starting Date (Crop Calendar)"""
plt.imshow(starting_date_rain, vmin= 0, vmax = 366)
plt.colorbar()
plt.title('Starting Date Rainfed')
plt.show()

plt.imshow(starting_date_irr, vmin= 0, vmax = 366)
plt.colorbar()
plt.title('Starting Date Irrigated')
plt.show()

"""Classified Yield"""
# plt.imshow(yield_map_rain_class)
# plt.colorbar()
# plt.title('Classified Rainfed Yield')
# plt.show()
# plt.imshow(yield_map_irr_class)
# plt.colorbar()
# plt.show('Classified Irrigated Yield')
# plt.show()

In [ ]:
# '''save result'''

obj_util.saveRaster(mask_path, './data_output/NB2/maiz_yield_rain.tif', yield_map_rain)
obj_util.saveRaster(mask_path, './data_output/NB2/maiz_yield_irr.tif', yield_map_irr)

obj_util.saveRaster(mask_path, './data_output/NB2/maiz_starting_date_rain.tif', starting_date_rain)
obj_util.saveRaster(mask_path, './data_output/NB2/maiz_starting_date_irr.tif', starting_date_irr)

# obj_utilities.saveRaster(mask_path, r'.\data_output\NB2\maiz_yld_rain_class.tif',yield_map_rain_class)
# obj_utilities.saveRaster(mask_path, r'.\data_output\NB2\maiz_yld_irr_class.tif',yield_map_irr_class)

### Thermal Screening Factor (fc1) Rainfed and Irrigated

In [ ]:
screen_factor = aez.getThermalReductionFactor()
fc1_rain = screen_factor[0]
fc1_irr = screen_factor[1]

In [ ]:
'''visualize additional outputs (Fc1)'''

plt.imshow(fc1_rain, vmax = 1, vmin = 0)
plt.colorbar()
plt.title('Fc1 Rainfed')
plt.show()

plt.imshow(fc1_irr, vmax = 1, vmin = 0)
plt.colorbar()
plt.title('Fc1 Irrigated')
plt.show()

In [ ]:
# '''save fc1 result'''
obj_util.saveRaster(mask_path, './data_output/NB2/fc1_maiz_rain.tif', fc1_rain)
obj_util.saveRaster(mask_path, './data_output/NB2/fc1_maiz_irr.tif', fc1_irr)

### Moisture Reduction Factor (fc2) (Only for Rainfed)

In [ ]:
fc2 = aez.getMoistureReductionFactor()

In [ ]:
""" visualize additional outputs (Fc2) """

plt.imshow(fc2, vmin= 0, vmax = 1)
plt.colorbar()
plt.title('fc2 (Moisture limited reduction factor)')
plt.show()

In [ ]:
# saving fc2 result
obj_util.saveRaster(mask_path, './data_output/NB2/fc2_maiz_rain.tif', fc2)

<hr>

### END OF MODULE 2: CROP SIMULATION

<hr>